# Exercise 2, Part A: RAG over the Wflow paper

Exercise 1 extracted structured fields from a hydrology PDF. This part stays in **text**: **Retrieval-Augmented Generation (RAG)** answers questions from a document you index yourself — here the Wflow_sbm GMD paper.

You will:

1. Load the paper and split it into overlapping chunks.
2. Build two retrievers on the **same** chunks: keyword (token overlap) and semantic (embeddings + cosine).
3. Compare what each method returns for hydrology queries.
4. Generate answers with Claude or Mistral: no context vs keyword vs semantic.

We do **not** compute RAGAS or other automatic RAG scores. Judge the chunks and answers yourself.

Prerequisites: course `uv sync` / hydrology-env verify is green, and Foundry (or Anthropic) variables are in the repo-root `.env`. Never paste API keys into this notebook.

## Step 1: Imports and paths


In [ ]:
import html
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from dllmforge.rag_preprocess_documents import PDFLoader, TextChunker
from IPython.display import Markdown, display

EXERCISE_DIR = Path.cwd()
if not (EXERCISE_DIR / "data" / "pdfs").exists():
    EXERCISE_DIR = Path.cwd() / "exercises" / "02_mcp_rag"
PDF_PATH = EXERCISE_DIR / "data" / "pdfs" / "gmd-17-3199-2024.pdf"
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Missing paper: {PDF_PATH}")

print(f"Exercise dir: {EXERCISE_DIR}")
print(f"PDF: {PDF_PATH.name}")


## Step 2: LLM and embeddings

`PROVIDER` picks the generator: `anthropic` (Claude) or `mistral`. Embeddings always use the Foundry OpenAI-compatible endpoint (`DEPLOYMENT_TEXT_EMBEDDINGS_3_LARGE`).

In [ ]:
PROVIDER = "anthropic"  # or "mistral"


def _load_env() -> None:
    env = find_dotenv(usecwd=True)
    if not env:
        for parent in [Path.cwd(), *Path.cwd().parents]:
            cand = parent / ".env"
            if cand.is_file():
                env = str(cand)
                break
    load_dotenv(env)


def foundry_anthropic_url() -> str | None:
    ep = (os.getenv("FOUNDRY_ENDPOINT") or "").strip().rstrip("/")
    if not ep or not os.getenv("FOUNDRY_KEY"):
        return None
    for suffix in ("/v1/messages", "/v1"):
        if ep.endswith(suffix):
            ep = ep[: -len(suffix)]
    return ep


def foundry_openai_url() -> str | None:
    if os.getenv("MISTRAL_ENDPOINT"):
        return os.environ["MISTRAL_ENDPOINT"].strip().rstrip("/")
    base = foundry_anthropic_url()
    if not base:
        return None
    return f"{base.split('/anthropic')[0]}/openai/v1"


def text_of(msg_or_content) -> str:
    # Claude returns a list of content blocks; keep only text.
    c = getattr(msg_or_content, "content", msg_or_content)
    if isinstance(c, str):
        return c
    parts = [
        b if isinstance(b, str) else b.get("text", "")
        for b in c or []
        if isinstance(b, str) or (isinstance(b, dict) and b.get("type") == "text")
    ]
    return "\n".join(p for p in parts if p)


def get_llm(provider: str):
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic

        model = (
            os.getenv("AGENT_ANTHROPIC_MODEL")
            or os.getenv("DEPLOYMENT_OPUS_55")
            or os.getenv("ANTHROPIC_DEPLOYMENT_SONNET55")
            or "claude-sonnet-5-5"
        )
        foundry_url = foundry_anthropic_url()
        if foundry_url:
            return ChatAnthropic(
                model=model, max_tokens=4096, max_retries=3,
                base_url=foundry_url, api_key=os.getenv("FOUNDRY_KEY"),
            )
        if os.getenv("ANTHROPIC_API_KEY"):
            return ChatAnthropic(model=model, max_tokens=4096, max_retries=3)
        raise RuntimeError("Set FOUNDRY_ENDPOINT + FOUNDRY_KEY, or ANTHROPIC_API_KEY.")
    if provider == "mistral":
        from langchain_openai import ChatOpenAI

        url = foundry_openai_url()
        key = os.getenv("MISTRAL_KEY") or os.getenv("FOUNDRY_KEY")
        model = (
            os.getenv("DEPLOYMENT_MISTRAL_LARGE_4")
            or os.getenv("DEPLOYMENT_MISTRAL_MEDIUM_35")
            or "mistral-large-4"
        )
        if not url or not key:
            raise RuntimeError(
                "Mistral needs FOUNDRY_ENDPOINT + FOUNDRY_KEY "
                "(or MISTRAL_ENDPOINT + MISTRAL_KEY)."
            )
        return ChatOpenAI(
            model=model, base_url=url, api_key=key, temperature=0.0,
            top_p=1, max_tokens=4096, max_retries=3, use_responses_api=False,
        )
    raise ValueError(f"Unknown PROVIDER {provider!r}; use 'anthropic' or 'mistral'.")


_load_env()
llm = get_llm(PROVIDER)

from langchain_openai import OpenAIEmbeddings

embed_url = foundry_openai_url()
if not embed_url or not os.getenv("FOUNDRY_KEY"):
    raise RuntimeError("Embeddings need FOUNDRY_ENDPOINT + FOUNDRY_KEY.")
embeddings = OpenAIEmbeddings(
    model=os.getenv("DEPLOYMENT_TEXT_EMBEDDINGS_3_LARGE", "text-embedding-3-large"),
    base_url=embed_url,
    api_key=os.getenv("FOUNDRY_KEY"),
    check_embedding_ctx_length=False,
)
print(f"LLM: {PROVIDER}")
print(f"Embeddings: {embeddings.model}")


## Step 3: Load the paper

`PDFLoader` (DLLMForge) extracts text per page with pypdf. Empty pages are skipped.

In [ ]:
loader = PDFLoader()
pages, file_name, _metadata = loader.load(PDF_PATH)
print(f"{file_name}: {len(pages)} pages with text")
print("First page preview:")
print(pages[0][1][:400])


## Step 4: Chunk the text

Two parameters matter:

- **`chunk_size`**: max characters per chunk. Smaller chunks retrieve more precisely but can split a sentence of context.
- **`overlap_size`**: characters copied from the end of one chunk into the next, so a split sentence is not lost.

We use the DLLMForge defaults (`1000` / `200`).

In [ ]:
chunker = TextChunker(chunk_size=1000, overlap_size=200)
chunks = chunker.chunk_text(pages, file_name)
print(f"{len(chunks)} chunks")
print(f"Example chunk (page {chunks[0]['page_number']}):")
print(chunks[0]["text"][:300])


## Step 5: Two indexes on the same chunks

**Keyword search** scores a chunk by how often query terms appear in it (after lowercasing, dropping stopwords, and treating `kv₀` as `kv0`). It is strong on symbols and exact phrases, weak on paraphrases. The score is a **raw hit count** (0, 1, 2, …) with **no upper bound**: it grows with how often the terms appear and with chunk length.

**Semantic search** embeds each chunk once, then ranks by cosine similarity to the query embedding. It is strong when the question uses different words than the paper ("soil permeability" vs saturated hydraulic conductivity). After L2-normalizing both vectors, cosine is theoretically **−1 to 1** (1 = same direction, 0 = orthogonal). Top hydrology hits usually sit in a **narrow positive band** (often roughly 0.2–0.6 with `text-embedding-3-large`), not near 1.

The two scores are **not on the same scale**. Rank within each list; do not compare a keyword `11` to a semantic `0.45`.

In [ ]:
_TOKEN = re.compile(r"[a-z0-9_]+", re.I)
_SUB = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
STOP = {"the", "a", "an", "is", "are", "was", "what", "how", "does", "do", "of", "in", "and", "or", "to", "for", "that"}


def tokenize(text: str) -> list[str]:
    return [t.lower() for t in _TOKEN.findall(text.translate(_SUB)) if len(t) > 1]


def keyword_search(query: str, docs: list[dict], k: int = 5) -> list[tuple[float, dict]]:
    q_terms = [t for t in tokenize(query) if t not in STOP]
    scored = []
    for doc in docs:
        hay = doc["text"].translate(_SUB).lower()
        score = float(sum(hay.count(t) for t in q_terms))
        scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    vecs: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        vecs.extend(embeddings.embed_documents(batch))
        print(f"embedded {min(i + batch_size, len(texts))}/{len(texts)}")
    return np.asarray(vecs, dtype=float)


def semantic_search(
    query: str, matrix: np.ndarray, docs: list[dict], k: int = 5
) -> list[tuple[float, dict]]:
    q = np.asarray(embeddings.embed_query(query), dtype=float)
    q = q / (np.linalg.norm(q) + 1e-12)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12
    scores = (matrix / norms) @ q
    order = np.argsort(-scores)[:k]
    return [(float(scores[i]), docs[i]) for i in order]


vectors = embed_texts([c["text"] for c in chunks])
print(f"vector matrix: {vectors.shape}")


## Step 6: Compare keyword vs semantic retrieval

Same `k=3`. Tables below: rank, score, page, snippet. Keyword scores are hit counts; semantic scores are cosine — do not compare the numbers across tables.

Typical split:

- `What is kv0?` — keyword often wins (exact symbol).
- `How permeable is the soil, and how does that control runoff?` — semantic often wins (paraphrase of Ksat / infiltration).
- `How is water routed down the hillslope and in the river?` — semantic often wins (same idea without the jargon).

In [ ]:
def hits_table(hits: list[tuple[float, dict]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "rank": i,
                "score": round(score, 3),
                "page": doc["page_number"],
                "snippet": " ".join(doc["text"].split())[:180],
            }
            for i, (score, doc) in enumerate(hits, 1)
        ]
    )


def show_rag(title: str, hits: list[tuple[float, dict]], answer: str | None = None) -> None:
    display(Markdown(f"## {title}"))
    display(hits_table(hits))
    for i, (score, doc) in enumerate(hits, 1):
        body = html.escape(doc["text"])
        display(
            Markdown(
                f"<details><summary><b>[{i}]</b> p.{doc['page_number']} · score {score:.3f}</summary>\n\n"
                f"<pre style='white-space:pre-wrap'>{body}</pre>\n\n</details>"
            )
        )
    if answer is not None:
        display(Markdown(f"### Answer\n\n{answer}"))


QUERIES = [
    "What is kv0?",
    "How permeable is the soil, and how does that control runoff?",
    "How is water routed down the hillslope and in the river?",
]
K = 3

retrieved = {}
for q in QUERIES:
    kw = keyword_search(q, chunks, k=K)
    sem = semantic_search(q, vectors, chunks, k=K)
    retrieved[q] = {"keyword": kw, "semantic": sem}
    display(Markdown(f"### Query: {q}"))
    display(Markdown("**Keyword** (hit count)"))
    display(hits_table(kw))
    display(Markdown("**Semantic** (cosine)"))
    display(hits_table(sem))


## Step 7: Generate answers

One paraphrase question, three generations:

1. **No context** — parametric memory only (may be fluent and uncited).
2. **Keyword chunks**.
3. **Semantic chunks**.

The system prompt forbids answering outside the retrieved context. If the chunks are wrong, a good model should say it does not know rather than invent Ksat values.

Expand a chunk (`details`) to read the text that actually went into the prompt.

In [ ]:
QUESTION = "How permeable is the soil, and how does that control runoff?"


def answer_with_chunks(question: str, hits: list[tuple[float, dict]]) -> str:
    context = "\n\n".join(
        f"[{doc['file_name']} p.{doc['page_number']}]\n{doc['text']}" for _, doc in hits
    )
    msg = llm.invoke(
        [
            (
                "system",
                "You are a hydrologist. Answer only from the provided context. "
                "If the context does not contain the answer, say you do not know. "
                "Cite file name and page.",
            ),
            ("human", f"Question: {question}\n\nContext:\n{context}"),
        ]
    )
    return text_of(msg)


def answer_no_context(question: str) -> str:
    msg = llm.invoke(
        [
            ("system", "You are a hydrologist."),
            ("human", question),
        ]
    )
    return text_of(msg)


kw_hits = retrieved[QUESTION]["keyword"]
sem_hits = retrieved[QUESTION]["semantic"]

display(Markdown(f"## No context\n\n{answer_no_context(QUESTION)}"))
show_rag("Keyword", kw_hits, answer_with_chunks(QUESTION, kw_hits))
show_rag("Semantic", sem_hits, answer_with_chunks(QUESTION, sem_hits))


## Step 8: Your turn

Change `MY_QUESTION` and re-run. Try a symbol (`kv0`, `Ksat`) and a paraphrase of the same idea. 

In [ ]:
MY_QUESTION = "What is the role of capillary rise in wflow_sbm?"

kw = keyword_search(MY_QUESTION, chunks, k=K)
sem = semantic_search(MY_QUESTION, vectors, chunks, k=K)
show_rag("Keyword", kw)
show_rag("Semantic", sem, answer_with_chunks(MY_QUESTION, sem))


## Takeaways

- **Keyword search** is good for codes and symbols that actually appear (`kv0`, `kinematic wave`).
- **Semantic search** is good when you ask in different words than the paper ("soil permeability" vs saturated hydraulic conductivity).